In [1]:
#libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)


In [2]:
#data loading

data1 = pd.read_csv('train.csv')
data2 = pd.read_csv('test.csv')

train_df = data1.copy()
test_df = data2.copy()

print("First 5 rows of train_df: \n", train_df.head())

print("First 5 rows of test_df: \n", test_df.head())

First 5 rows of train_df: 
    id  annual_income  debt_to_income_ratio  credit_score  loan_amount  \
0   0       29367.99                 0.084           736      2528.42   
1   1       22108.02                 0.166           636      4593.10   
2   2       49566.20                 0.097           694     17005.15   
3   3       46858.25                 0.065           533      4682.48   
4   4       25496.70                 0.053           665     12184.43   

   interest_rate  gender marital_status education_level employment_status  \
0          13.67  Female         Single     High School     Self-employed   
1          12.92    Male        Married        Master's          Employed   
2           9.76    Male         Single     High School          Employed   
3          16.10  Female         Single     High School          Employed   
4          10.21    Male        Married     High School          Employed   

         loan_purpose grade_subgrade  loan_paid_back  
0              

In [3]:
train_df = train_df.drop(columns=['id'], axis=1)

test_id_placeholder = test_df['id']
test_df = test_df.drop(columns=['id'], axis=1)

In [4]:
#feature engineering

train_df['in_relationship'] = (train_df['marital_status'] == 'Married').astype(int)
test_df['in_relationship'] = (test_df['marital_status'] == 'Married').astype(int)

train_df['has_studied'] = train_df['education_level'].isin(["Bachelor's", "High School", "Master's", "PhD"]).astype(int)
test_df['has_studied'] = test_df['education_level'].isin(["Bachelor's", "High School", "Master's", "PhD"]).astype(int)

train_df['is_employed'] = train_df['employment_status'].isin(['Employed', 'Self-Employed']).astype(int)
test_df['is_employed'] = test_df['employment_status'].isin(['Employed', 'Self-Employed']).astype(int)

train_df['in_debt'] = (train_df['loan_purpose'] == 'Debt consolidation').astype(int)
test_df['in_debt'] = (test_df['loan_purpose'] == 'Debt consolidation').astype(int)

In [5]:
gender_map = {'Male': 0, 'Female': 1, 'Other': 2}

train_df['gender'] = train_df['gender'].map(gender_map)
test_df['gender'] = test_df['gender'].map(gender_map)

In [6]:
train_df = train_df.drop(columns=['marital_status', 'education_level', 'employment_status', 'loan_purpose'])
test_df = test_df.drop(columns=['marital_status', 'education_level', 'employment_status', 'loan_purpose'])

In [7]:
from category_encoders import TargetEncoder

encoder = TargetEncoder()

train_df['grade_encoded'] = encoder.fit_transform(
    train_df['grade_subgrade'],
    train_df['loan_paid_back']
)

test_df['grade_encoded'] = encoder.transform(
    test_df['grade_subgrade']
)

In [8]:
train_df = train_df.drop(columns='grade_subgrade')
test_df = test_df.drop(columns='grade_subgrade')

In [9]:
target_placeholder = train_df.pop('loan_paid_back')

train_df['loan_paid_back'] = target_placeholder

In [10]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X = train_df.drop(columns='loan_paid_back')
y = train_df['loan_paid_back']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#residual boosting 

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBRegressor

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
rf_probs = rf_model.predict_proba(X_train)[:,1] #what's the probability of 1

residuals = y_train - rf_probs #actual vs predicted probability

In [14]:
#XGB on residuals
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, residuals) #xgb specializing on the errors of RandomForest

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [18]:
#final prediction

rf_test_probs = rf_model.predict_proba(X_test)[:,1]
xgb_residual_preds = xgb_model.predict(X_test)

In [19]:
final_probs = rf_test_probs + xgb_residual_preds

final_probs = (final_probs > 0.5).astype(int)

In [22]:
#predictions on test_df
rf_test_probs = rf_model.predict_proba(test_df)[:, 1]
xgb_residual_preds = xgb_model.predict(test_df)

final_probs = rf_test_probs + xgb_residual_preds

#clipping to avoid results different from 0 and 1
final_probs = np.clip(final_probs, 0, 1)

final_preds = (final_probs > 0.5).astype(int)

In [ ]:
# submission_df = pd.DataFrame({
#     "id": test_id_placeholder,
#     "loan_paid_back": final_preds
# })

# submission_df.to_csv('submission5.csv',index=False)
# print("Success!")

Success!
